# Análises, plotagens e *insights*
 Esse *notebook* contém gráficos e *insights* obtidos a partir do *Dataframe* que foi limpo no *notebook* anterior.



## Parte I - Carregamento dos dados e importações
 
 Muito similar a sessão de mesma parte no *notebook* anterior, porém desta vez estou apenas checando se tudo feito lá funcionou, visto que já sei sobre os dados. Para evitar eventuais conflitos com o arquivo anterior chamarei o *dataframe* de **df_limpo**; O mesmo vale para outros que possam aparecer posteriormente, mas sempre irei explicar seu *alias*.

In [15]:
# Importação das bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


In [16]:
# Abrindo o arquivo e transformando em um dataframe

df_limpo = pd.read_csv(r"C:\Users\piaze\Documents\Projetos\Exercícios\EDA-Carros-Eletricos-Washington\base_de_dados\ev_dados_washington_limpo.csv")
df_limpo.head(10)

,County,City,Model Year,Make,Model,Electric Vehicle Type,CAFV Status,Electric Range,Electric Utility,Vehicle Age
0,Kitsap,Bainbridge Island,2018,TESLA,MODEL 3,BEV,Eligible,215.0,Puget Sound Energy,8
1,Kitsap,Port Orchard,2011,NISSAN,LEAF,BEV,Eligible,73.0,Puget Sound Energy,15
2,King,Bothell,2011,NISSAN,LEAF,BEV,Eligible,73.0,Puget Sound Energy,15
3,Yakima,Selah,2020,CHEVROLET,BOLT EV,BEV,Eligible,259.0,Pacificorp,6
4,King,Seattle,2020,KIA,NIRO,PHEV,Not Eligible (Low Range),26.0,City Of Seattle,6
5,Thurston,Olympia,2023,KIA,NIRO,PHEV,Eligible,33.0,Puget Sound Energy,3
6,King,Seattle,2016,AUDI,A3,PHEV,Not Eligible (Low Range),16.0,City Of Seattle,10
7,Thurston,Tumwater,2018,TESLA,MODEL 3,BEV,Eligible,215.0,Puget Sound Energy,8
8,Kitsap,Bremerton,2018,TESLA,MODEL S,BEV,Eligible,249.0,Puget Sound Energy,8
9,Snohomish,Lynnwood,2017,CHEVROLET,VOLT,PHEV,Eligible,53.0,Puget Sound Energy,9


## Parte II - Visão geral da frota

 Esta parte contém gráficos com informações gerais da frota, com base em todas as 270 mil linhas do *datafreame*, entre elas:

 **1 -** Top 10 marcas entre a frota, qual marca tem mais carros registrados?

 **2 -** BEV VS PHEV qual domina a frota?

 **3 -** Qual idade média da frota?

 **4 -** Qual a relação entre carros que podem usar *CAFV* e os que não?

 **5 -** E por fim, quais cidades tem a maior concentração de veículos?

In [17]:
# Separando as top 10 marcas

df_marcas = df_limpo["Make"].value_counts().reset_index()

df_marcas.columns = ["Marca", "Quantidade"]

top_10_marcas = df_marcas.head(10)


top_10_marcas

,Marca,Quantidade
0,TESLA,110210
1,CHEVROLET,19015
2,NISSAN,15938
3,FORD,14908
4,KIA,13600
5,TOYOTA,11350
6,BMW,11176
7,HYUNDAI,9806
8,RIVIAN,8491
9,VOLKSWAGEN,7356


In [18]:
# Top 10 marcas em gráfico
# Percebe-se um amplo dominio da Tesla

fig = px.bar(top_10_marcas, 
            x="Quantidade", 
            y="Marca", 
            title="<b>Figura 1: Top 10 marcas na frota<b>", 
            width=800, 
            height=600, 
            color="Marca", 
            color_discrete_sequence=px.colors.qualitative.D3)

fig.update_layout(title_x=0.5)

fig.show()

### Análise da figura 1
 
 Apesar de ser uma estatística muito simples, ela entrega muito sobre o panorama da frota, e até questionamentos sobre o mercado; Como: Apesar da sua recente tradição no mercado Norte-Americano porque a Tesla tem uma vantagem tão grande? A chinesa Rivian que "Surgiu" recentemente no mercado americano parece ter um crescimento rápido e interessante.

In [19]:
# Divisão da frota, BEVs vs PHEVs

bev_vs_phev = df_limpo["Electric Vehicle Type"].value_counts().reset_index()
bev_vs_phev.columns = ["Tipo", "Quantidade"]

bev_vs_phev

,Tipo,Quantidade
0,BEV,215829
1,PHEV,54625


In [20]:
# Gráfico de pizza comparando os dois

fig = px.pie(bev_vs_phev, 
            values="Quantidade", 
            names="Tipo", 
            title="<b>Figura 2: Proporção entre BEVs e PHEVs na frota<b>", 
            width=600,
            height=400,     
            color='Tipo',  
            color_discrete_map={"BEV": "#5B8DB8", "PHEV": "#B0B0B0"})

fig.update_layout(title_x=0.5, legend_title="Tipo")

fig.show()

### Análise da figura 2
 A imagem fala por si só, os BEVs dominam o mercado, e com uma rápida pesquisa em algum motor de busca ou até mesmo o site do governo de Washington, se percebe o porque: 1: O BEVs são mais acessíveis e tem isenções em certos impostos e 2: Por serem 100% elétricos sua mecânica é mais simples, sem um motor a combustão extra ou peças extras, o que gera um preço menor e maior facilidade na manutenção; Apesar de cada um ter suas vantagens específicas.

In [21]:
# Média de idade da frota

df_limpo["Vehicle Age"].mean().round()

np.float64(4.0)

In [22]:
# Agrupando por ano
# Com isso podemos chegar a duas possíveis conclusões, 1: as pessoas estão adotando EVs com um boom definitivo a partir de 2020 
# ou 2: As pessoas trocam de carro com frequência; logo não sabemos de seu veículo anterior já era elétrico.

evolucao_anual = df_limpo["Model Year"].value_counts().sort_index().reset_index()
evolucao_anual.columns = ["Ano", "Quantidade"]

fig = px.area(evolucao_anual, 
            x="Ano", 
            y="Quantidade",
            title="<b>Curva de idade entre a frota<b>", 
            width=1000, 
            height=500)
             

fig.update_traces(line_color='#1f77b4')

fig.show()

In [23]:
df_cafv = df_limpo.groupby(['Electric Vehicle Type', 'CAFV Status']).size().reset_index(name='Contagem')

df_cafv["CAFV Status"] = df_cafv["CAFV Status"].map({"Eligible": "Elegível","Not Eligible (Low Range)": "Não Elegível (Baixa Autonomia)"})

df_cafv

,Electric Vehicle Type,CAFV Status,Contagem
0,BEV,Elegível,45391
1,BEV,Não Elegível (Baixa Autonomia),8
2,PHEV,Elegível,30632
3,PHEV,Não Elegível (Baixa Autonomia),23993


In [24]:
# Relação entre CAFV

fig = px.bar(df_cafv, 
             x="Electric Vehicle Type", 
             y="Contagem", 
             color="CAFV Status",
             title="<b>Correlação entre Tipologia Tecnológica e Elegibilidade CAFV</b>",
             barmode="stack",
             template="plotly_white")

fig.update_layout(xaxis_title="Tipo de Veículo", yaxis_title="Volume de Registros", legend_title="Status CAFV")

fig.show()


In [25]:
# Cidades com mais carros

cidades = df_limpo["City"].value_counts().reset_index()
cidades.columns = ["Cidade", "Quantidade"]

cidades

,Cidade,Quantidade
0,Seattle,42105
1,Bellevue,13120
2,Vancouver,10136
3,Redmond,9242
4,Bothell,8936
...,...,...
489,Danville,1
490,Matlock,1
491,Artondale,1
492,Farmington,1


In [26]:
# Plotando

fig = px.bar(cidades.head(15), 
            x="Quantidade",    
            y="Cidade", 
            width=800, 
            height=600, 
            title="<b>Top 15 cidades com mais veículos registrados<b>",
            color_discrete_sequence=["#1f77b4"])

fig.update_layout(title_x=0.5)
fig.update_yaxes(autorange="reversed")

fig.show()